# Import dependencies

In [1]:
import numpy as np
import pandas as pd

print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

NumPy: 2.5.4
pandas: 3.1.1


# Load dataset

In [2]:
# Since the dataset file is saved in the Strict Open XML Spreadsheet format, we
# have to use `calamine` to read it
df = pd.read_excel("dataset/dataset-uci.xlsx", engine="calamine")

# Shape of dataset; data types; feature overview; missing values; duplicates; etc.
rows = len(df)
print(f"Number of rows: {rows}")
#how many columns we have of data
columns = len(df.columns)
print(f"Number of columns: {columns}")
##Now the actual shape of the dataset + types
print(f"Dataset shape: {df.shape}")
print(f"Dataset datatypes:\n{df.dtypes}")

print(f"# null values for each feature:\n{df.isnull().sum()}")
print(f"# null values: {df.isnull().sum().sum()}")

print(f"# duplicated values: {df.duplicated().sum()}")

y = df["Gallstone Status"]
X = df.drop(columns="Gallstone Status")

Number of rows: 319
Number of columns: 39
Dataset shape: (319, 39)
Dataset datatypes:
Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)             

We have data of around 319 ppl.

In [3]:
#How many ppl have gallstones and how many do not
gallstone_counts = y.value_counts()[1]
print(f"Number of people with gallstones: {gallstone_counts}")
print(f"Number of people without gallstones: {len(X) - gallstone_counts}")


Number of people with gallstones: 158
Number of people without gallstones: 161


So we have half half ppl with and without gallstones

In [4]:
# Outlier
v_max =  X.max()
v_mean = X.mean()
v_min =  X.min()
v_std =  X.std()

z = (X - v_mean) / v_std
outliers = z.abs() > 10

print(outliers.sum())               # number of outliers per column
result = pd.DataFrame({
    "value": X[outliers].stack().dropna(),
    "z": z[outliers].stack().dropna(),
})

feature = result.index.get_level_values(1)
result["mean"] = v_mean[feature].values
result["std"] = v_std[feature].values

result = result[["value", "mean", "std", "z"]].rename_axis(["Record", "Feature"])
print(result.round(2))

Age                                               0
Gender                                            0
Comorbidity                                       0
Coronary Artery Disease (CAD)                     0
Hypothyroidism                                    0
Hyperlipidemia                                    0
Diabetes Mellitus (DM)                            0
Height                                            0
Weight                                            0
Body Mass Index (BMI)                             0
Total Body Water (TBW)                            0
Extracellular Water (ECW)                         0
Intracellular Water (ICW)                         0
Extracellular Fluid/Total Body Water (ECF/TBW)    0
Total Body Fat Ratio (TBFR) (%)                   0
Lean Mass (LM) (%)                                0
Body Protein Content (Protein) (%)                0
Visceral Fat Rating (VFR)                         0
Bone Mass (BM)                                    0
Muscle Mass 

In [5]:
# This is very high but clinically possible in poorly controlled diabetes
X.loc[40, "Diabetes Mellitus (DM)"]
X = X.drop(40)
y = y.drop(40)

In [6]:
# This isn’t physiologically possible. It may be a misplaced decimal, such as 19.54 or 195.4. Obesity % is usually derived from BMI
X.loc[239, ["Height", "Weight", "Body Mass Index (BMI)", "Obesity (%)"]]
X.loc[239, "Obesity (%)"] = 19.54

In [7]:
X.loc[318, ["Total Cholesterol (TC)", "Low Density Lipoprotein (LDL)", "High Density Lipoprotein (HDL)"]]
X = X.drop(318)
y = y.drop(318)

In [8]:
# Highly correlated features
corr = X.corr().abs()
upper = corr.where(np.triu(
    np.ones(corr.shape, dtype=bool),
    k=1
))  # upper triangle only

pairs = upper.stack()
high = pairs[pairs > 0.9]
print(high.sort_values(key=np.abs, ascending=False))


Total Body Fat Ratio (TBFR) (%)  Lean Mass (LM) (%)           0.994034
Total Body Water (TBW)           Muscle Mass (MM)             0.928283
Body Mass Index (BMI)            Obesity (%)                  0.917529
Total Body Water (TBW)           Extracellular Water (ECW)    0.904031
Intracellular Water (ICW)        Muscle Mass (MM)             0.903599
Body Mass Index (BMI)            Total Fat Content (TFC)      0.902578
dtype: float64


In [9]:
keep = []
for c in X.columns:
    if not (corr.loc[c, keep] > 0.9).any():
        keep.append(c)

dropped = [c for c in X.columns if c not in keep]
print(f"Dropped {len(dropped)}:", dropped)

X = X[keep]

Dropped 5: ['Extracellular Water (ECW)', 'Lean Mass (LM) (%)', 'Muscle Mass (MM)', 'Obesity (%)', 'Total Fat Content (TFC)']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [11]:
# Continuous features only (binary and ordinal ones have ≤ 5 distinct values)
cont = [c for c in X_train.columns if X_train[c].nunique() > 5]

skew = X_train[cont].skew()
skewed = skew[skew > 1].index.tolist()                   # strongly right-skewed
skewed = [c for c in skewed if (X_train[c] >= 0).all()]  # log1p needs x > -1

skew_table = pd.DataFrame({
    "Skewness": skew[skewed],
    "After log": np.log1p(X_train[skewed]).skew(),
}).sort_values("Skewness", ascending=False).round(2)

print(skew_table)
print(skew_table.to_latex())

                                Skewness  After log
Aspartat Aminotransferaz (AST)      6.57       1.67
C-Reactive Protein (CRP)            5.16       1.63
Glucose                             4.40       2.72
Alanin Aminotransferaz (ALT)        3.98       0.66
Triglyceride                        2.83      -0.82
Visceral Fat Area (VFA)             1.24      -0.73
Intracellular Water (ICW)           1.09       0.18
\begin{tabular}{lrr}
\toprule
 & Skewness & After log \\
\midrule
Aspartat Aminotransferaz (AST) & 6.570000 & 1.670000 \\
C-Reactive Protein (CRP) & 5.160000 & 1.630000 \\
Glucose & 4.400000 & 2.720000 \\
Alanin Aminotransferaz (ALT) & 3.980000 & 0.660000 \\
Triglyceride & 2.830000 & -0.820000 \\
Visceral Fat Area (VFA) & 1.240000 & -0.730000 \\
Intracellular Water (ICW) & 1.090000 & 0.180000 \\
\bottomrule
\end{tabular}



In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import FunctionTransformer, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def make_model(C):
    #return make_pipeline(
    #    StandardScaler(),
    #    LogisticRegression(C=C, max_iter=5000),
    #)
    return make_pipeline(
        ColumnTransformer([("log", FunctionTransformer(np.log1p), skewed)],
                          remainder="passthrough"),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=5000),
    )


# 2. Choose C with cross-validation on the training set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    scores = cross_validate(make_model(C), X_train, y_train, cv=cv,
                            scoring=["accuracy", "roc_auc", "f1"])
    results[C] = scores["test_roc_auc"].mean()

    print(f"C={C}:")
    for metric in ["accuracy", "roc_auc", "f1"]:
        s = scores[f"test_{metric}"]
        print(f"{metric:>8}: {s.mean():.3f} ± {s.std():.3f}")

best_C = max(results, key=results.get)
print(f"\nBest C (by CV ROC AUC): {best_C}")

# 3. Refit on the full training set and evaluate once on the test set
final_model = make_model(best_C)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print("\nTest set performance:")
print(f"accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f" roc_auc: {roc_auc_score(y_test, y_prob):.3f}")
print(f"      f1: {f1_score(y_test, y_pred):.3f}")

C=0.001:
accuracy: 0.700 ± 0.077
 roc_auc: 0.800 ± 0.066
      f1: 0.656 ± 0.106
C=0.01:
accuracy: 0.787 ± 0.054
 roc_auc: 0.870 ± 0.057
      f1: 0.766 ± 0.067
C=0.1:
accuracy: 0.834 ± 0.072
 roc_auc: 0.888 ± 0.058
      f1: 0.825 ± 0.076
C=1:
accuracy: 0.823 ± 0.048
 roc_auc: 0.888 ± 0.052
      f1: 0.813 ± 0.057
C=10:
accuracy: 0.838 ± 0.037
 roc_auc: 0.883 ± 0.055
      f1: 0.829 ± 0.047
C=100:
accuracy: 0.819 ± 0.048
 roc_auc: 0.879 ± 0.058
      f1: 0.812 ± 0.053
C=1000:
accuracy: 0.822 ± 0.042
 roc_auc: 0.877 ± 0.061
      f1: 0.814 ± 0.050

Best C (by CV ROC AUC): 0.1

Test set performance:
accuracy: 0.812
 roc_auc: 0.877
      f1: 0.800


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FunctionTransformer, make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "logreg": make_pipeline(
        StandardScaler(),
        LogisticRegression(C=10, max_iter=5000),
    ),
    "logreg + log": make_pipeline(
        ColumnTransformer([("log", FunctionTransformer(np.log1p), skewed)],
                          remainder="passthrough"),
        StandardScaler(),
        LogisticRegression(C=1, max_iter=5000),
    ),
    "random forest": make_pipeline(
        RandomForestClassifier(n_estimators=500, random_state=42),
    ),
    # "gradient boosting": HistGradientBoostingClassifier(   # handles NaN natively
    #     max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}

In [14]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

rows = []
for name, model in models.items():
    # Validation (training set only)
    s = cross_val_score(model, X_train, y_train, cv=rcv, scoring="roc_auc")

    # Test (fit once on the full training set, evaluate once)
    model.fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    rows.append({
        "Model": name,
        "CV ROC AUC": f"${s.mean():.3f} \\pm {s.std():.3f}$",
        "Test ROC AUC": f"{test_auc:.3f}",
    })

results = pd.DataFrame(rows)
print(results)
print(results.to_latex(index=False, column_format="lcc"))

           Model         CV ROC AUC Test ROC AUC
0         logreg  $0.864 \pm 0.041$        0.854
1   logreg + log  $0.889 \pm 0.035$        0.897
2  random forest  $0.849 \pm 0.058$        0.861
\begin{tabular}{lcc}
\toprule
Model & CV ROC AUC & Test ROC AUC \\
\midrule
logreg & $0.864 \pm 0.041$ & 0.854 \\
logreg + log & $0.889 \pm 0.035$ & 0.897 \\
random forest & $0.849 \pm 0.058$ & 0.861 \\
\bottomrule
\end{tabular}



In [17]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone

summaries = {}

for name, model in models.items():
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

    fold_imp = []
    for tr, va in cv.split(X_train, y_train):
        m = clone(model).fit(X_train.iloc[tr], y_train.iloc[tr])
        r = permutation_importance(m, X_train.iloc[va], y_train.iloc[va],
                                   scoring="roc_auc", n_repeats=10, random_state=42)
        fold_imp.append(r.importances_mean)

    imp = pd.DataFrame(fold_imp, columns=X_train.columns)
    summaries[name] = pd.DataFrame({
        "mean": imp.mean(),
        "std": imp.std(),
        "top5_rate": (imp.rank(axis=1, ascending=False) <= 5).mean(),
    }).sort_values("mean", ascending=False)
    print(f"\n{name}")
    print(summaries[name].head(10).round(3))

features = ["C-Reactive Protein (CRP)", "Vitamin D", "Bone Mass (BM)",
            "Extracellular Fluid/Total Body Water (ECF/TBW)", "Gender",
            "Total Body Fat Ratio (TBFR) (%)"]

table = pd.DataFrame({name: s["top5_rate"] for name, s in summaries.items()})
table["Sum"] = table.sum(axis=1)
table = table.sort_values("Sum", ascending=False)

print(table.head(8).round(2))
print(table.head(8).round(2).to_latex())


logreg
                                                 mean    std  top5_rate
Bone Mass (BM)                                  0.137  0.047       1.00
Total Body Fat Ratio (TBFR) (%)                 0.135  0.074       0.80
Intracellular Water (ICW)                       0.084  0.051       0.68
C-Reactive Protein (CRP)                        0.083  0.043       0.56
Gender                                          0.083  0.037       0.68
Visceral Fat Rating (VFR)                       0.059  0.030       0.40
Aspartat Aminotransferaz (AST)                  0.053  0.024       0.36
Vitamin D                                       0.050  0.019       0.12
Extracellular Fluid/Total Body Water (ECF/TBW)  0.042  0.023       0.08
Hemoglobin (HGB)                                0.034  0.019       0.04

logreg + log
                                                 mean    std  top5_rate
C-Reactive Protein (CRP)                        0.135  0.031       1.00
Bone Mass (BM)                            

In [18]:
from sklearn.feature_selection import f_classif
from scipy.stats import mannwhitneyu

F, p = f_classif(X, y)

mw_p = [mannwhitneyu(X.loc[y == 1, c],
                     X.loc[y == 0, c]).pvalue
        for c in X.columns]

uni = pd.DataFrame({"F": F, "p_anova": p, "p_mannwhitney": mw_p},
                   index=X.columns).sort_values("F", ascending=False)
print(uni.head(10).round(4).to_latex())

\begin{tabular}{lrrr}
\toprule
 & F & p_anova & p_mannwhitney \\
\midrule
Vitamin D & 45.729100 & 0.000000 & 0.000000 \\
C-Reactive Protein (CRP) & 27.430500 & 0.000000 & 0.000000 \\
Total Body Fat Ratio (TBFR) (%) & 16.370600 & 0.000100 & 0.000100 \\
Bone Mass (BM) & 14.941400 & 0.000100 & 0.000100 \\
Hemoglobin (HGB) & 12.487100 & 0.000500 & 0.000800 \\
Extracellular Fluid/Total Body Water (ECF/TBW) & 9.893000 & 0.001800 & 0.013700 \\
High Density Lipoprotein (HDL) & 9.180300 & 0.002600 & 0.002300 \\
Hyperlipidemia & 8.536400 & 0.003700 & 0.003900 \\
Gender & 7.068000 & 0.008200 & 0.008500 \\
Visceral Fat Area (VFA) & 6.479900 & 0.011400 & 0.004800 \\
\bottomrule
\end{tabular}

